## CLEANING PHASE

Step 1 - Load the data and do an inspection first

In [1]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

df = pd.read_csv("../data/raw_data.csv")

print(df.shape)

#shows you column dtypes, non-null counts per column
df.info()
df.head(3)

#clean per-column count of missing values
df.isnull().sum()

(418, 8)
<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   title        418 non-null    str  
 1   company      418 non-null    str  
 2   location     366 non-null    str  
 3   tags         418 non-null    str  
 4   description  418 non-null    str  
 5   apply_url    418 non-null    str  
 6   date_posted  418 non-null    str  
 7   search_tag   418 non-null    str  
dtypes: str(8)
memory usage: 26.3 KB


title           0
company         0
location       52
tags            0
description     0
apply_url       0
date_posted     0
search_tag      0
dtype: int64

Step 2 - Handle Missing values

In [3]:
df["company"] = df["company"].fillna("Not specified")
df["location"] = df["location"].fillna("Not specified")
df["tags"] = df["tags"].fillna("")
df["description"] = df["description"].fillna("")

Step 3 - Drop duplicate postings

In [4]:
before = len(df)
df = df.drop_duplicates(subset=["title", "company", "apply_url"]) #The apply URL is unique per posting
after = len(df)
print(f"Removed {before - after} duplicate rows")

Removed 0 duplicate rows


Step 4 - Lowercase and strip text columns

In [5]:
text_cols = ["title", "company", "location", "tags", "description"]

#.astype(str) guarantees that every value is text
for col in text_cols:
    df[col] = df[col].astype(str).str.lower().str.strip()

Step 5 - Remove leftover HTML/whitespace artifacts

In [6]:
import re

def clean_text_artifacts(text):
    text = re.sub(r"\s+", " ", text)      # collapse multiple spaces/newlines into one
    text = text.replace("\xa0", " ")       # remove non-breaking space characters
    text = text.strip()
    return text

for col in text_cols:
    df[col] = df[col].apply(clean_text_artifacts)

Step 6 - Check the result

In [26]:
df[["title", "company", "location", "description"]].sample(5)

,title,company,location,description
254,social media and influencer manager,"pete &amp; gerry's organics, llc","chicago, chicago, illinois, united states","job type full-time description healthy hens, h..."
127,team leader housekeeping,hyatt regency,"dehradun,",organization- hyatt regency dehradun summary y...
233,associate account strategist donut studios,new engen,"new york, new york, new york, united states","why donut studios? at new engen, we help brand..."
369,product marketing intern,everlaw,oakland,"as a product marketing intern at everlaw, you ..."
201,senior data engineer,lalamove,kuala lumpur,"at lalamove, we believe in the power of commun..."


In [9]:
print((df["description"].str.len() < 20).sum(), "postings have very short descriptions")

0 postings have very short descriptions


Step 7 - Save the cleaned version

In [10]:
df.to_csv("../data/cleaned_data.csv", index=False)
print(f"Final cleaned dataset: {len(df)} rows")

Final cleaned dataset: 418 rows


## SKILLS EXTRACTION

Step 1 - Load you cleaned data

In [29]:
import pandas as pd
import re

df = pd.read_csv("../data/cleaned_data.csv")
df.shape

(418, 8)

Step 2 - Define you skill list, grouped by category

In [30]:
technical_skills = [
    "python", "sql", "r", "java", "javascript", "c++", "c#", "go", "scala",
    "excel", "power bi", "tableau", "looker",
    "aws", "azure", "gcp", "docker", "kubernetes",
    "machine learning", "deep learning", "nlp", "statistics",
    "spark", "hadoop", "airflow", "git", "linux",
    "figma", "photoshop", "seo", "html", "css"
]

soft_skills = [
    "communication", "collaboration", "teamwork", "leadership",
    "problem solving", "stakeholder management", "critical thinking",
    "time management", "adaptability", "presentation"
]